In [1]:
import os
from pathlib import Path

import ee
import geopandas as gpd
import numpy as np
import rasterio as rio
import rasterio.transform as rio_transform
from scipy.interpolate import griddata

from belo_horizonte.bounds import load_bounds
from belo_horizonte.temperature import get_lst
from belo_horizonte.utils import clamp_bounds

c:\Users\lain\Documents\belo_horizonte\.venv\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
ee.Initialize()

In [3]:
data_path = Path("./data")
generated_path = Path(os.environ["DATA_PATH"]) / "generated"

In [4]:
df_level = (
    gpd.read_file(
        data_path / "CURVA_DE_NIVEL_5M.zip",
    )
    .filter(["COTA", "geometry"])
    .rename(columns={"COTA": "elevation"})
)

In [8]:
sampling_dist = 50
points, elevations = [], []

for _, row in df_level.iterrows():
    geom = row.geometry
    distances = np.arange(0, geom.length, sampling_dist)
    for d in distances:
        pt = geom.interpolate(d)
        points.append([pt.x, pt.y])
        elevations.append(row.elevation)

points = np.array(points)
elevations = np.array(elevations)

In [11]:
resolution = 30
bounds = clamp_bounds(*df_level.total_bounds, scale=30)

xi = np.arange(bounds[0], bounds[2], resolution)
yi = np.arange(bounds[1], bounds[3], resolution)
xi, yi = np.meshgrid(xi, yi)

zi = griddata(points, elevations, (xi, yi), method="linear")

In [12]:
transform = rio_transform.from_bounds(*bounds, height=zi.shape[0], width=zi.shape[1])
with rio.open(
    generated_path / "elevation.tif",
    "w",
    driver="GTiff",
    height=zi.shape[0],
    width=zi.shape[1],
    count=1,
    dtype=zi.dtype,
    crs=df_level.crs,
    transform=transform,
    compress="lzw",
) as dst:
    dst.write(np.flip(zi, axis=0), 1)